In [2]:
import os
import sys
import datetime
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

sys.path.insert(0, '../../retrosheet')
sys.path.insert(0, '../../pybaseball')

from retrosheet import Retrosheet # https://github.com/calestini/retrosheet
import pybaseball as pbb # https://github.com/jldbc/pybaseball
pbb.cache.enable()

import psycopg2
from sqlalchemy import create_engine

In [5]:
class Database:
    """"""
    def __init__(self):
        self.creds = {
            'database': 'baseball',
            'user': 'tjordan',
            'password': '',
            'host': '127.0.0.1',
            'port': '5432',
        }

        self.engine = create_engine('postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}'.format(**self.creds))
        self.conn = psycopg2.connect(**self.creds)
        self.conn.autocommit = True
        self.cursor = self.conn.cursor()

In [6]:
db = Database()

In [2]:
class RetroETL:
    def __init__(self):
        self.SEASON_RANGE = (2017,2017)
        self.GAME_LOOKBACK = [None,3,6,9,12,20,50]
        self.ATBAT_LOOKBACK = [None,20,50,100,300,600]
        
        self.rs = Retrosheet()
        self.retro_fpath = os.path.join(os.getcwd(),'retrosheet')
        self.retro_datasets = [
            'batting','fielding','info','lineup',
            'pitching','plays','rosters','running','teams',
        ]

        
    def setup(self,cursor):
        os.makedirs(self.retro_fpath)

        cursor.execute('''create schema base''')
        cursor.execute('''create schema proc''')
    
    def extract(self):
        try:
            os.chdir(self.retro_fpath)
            self.rs.batch_parse(
                yearFrom=self.SEASON_RANGE[0],
                yearTo=self.SEASON_RANGE[1],
                batchsize=10
            )
        except:
            raise
        finally:
            os.chdir(self.retro_fpath+'/..')
    
    def load(self):
        for dataset in self.retro_datasets:
            setattr(self,f'DF_{dataset.upper()}',pd.read_csv(f'{self.retro_fpath}/{dataset}.csv'))

    def transform(self):
        None
    
    def write(self,engine):
        for table,data in self.retro_data.items():
            data.to_sql(
                table.replace('RS_','').lower(),engine,
                schema='base',#database='baseball',
                if_exists='replace',index=False,
            )

In [3]:
retro = RetroETL()
# retro.setup(db.cursor)
retro.extract()
retro.load()
retro.transform()
# retro.write(db.engine)

[============================================================] 100.0% ... Completed 2017-2017


In [4]:
retro.PLAYS

,game_id,order,pitcher,pitch_count,inning,team,player_id,count_on_batter,pitch_str,play_str,B,1,2,3,H,run,out,on-B,on-1,on-2,on-3,on-H,hometeam_score,awayteam_score,trajectory,passes,location,pre_state,post_state,play_runs,play_outs
0,ANA201704070,0,chavj001,6,1,0,seguj002,32,CBSBBB,W,0,1,0,0,0,0,0,NaN,seguj002,NaN,NaN,[],0,0,NaN,[],NaN,1,2,0,0
1,ANA201704070,1,chavj001,12,1,0,hanim001,12,1CB1FT,K,0,1,0,0,0,0,1,NaN,seguj002,NaN,NaN,[],0,0,NaN,[],NaN,2,10,0,1
2,ANA201704070,2,chavj001,15,1,0,canor001,11,BFX,8/F,0,1,0,0,0,0,2,NaN,seguj002,NaN,NaN,[],0,0,F,['8'],NaN,10,18,0,1
3,ANA201704070,3,chavj001,18,1,0,cruzn002,1,C1>S,SB2,0,0,1,0,0,0,2,cruzn002,NaN,seguj002,NaN,[],0,0,NaN,[],NaN,18,19,0,0
4,ANA201704070,4,chavj001,21,1,0,cruzn002,22,C1>S.*BBS,K,0,0,1,0,0,0,3,NaN,NaN,seguj002,NaN,[],0,0,NaN,[],NaN,19,25,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188901,WAS201709300,74,kintb001,22,9,0,morom001,0,X,T8/L.3-H;2-H;1-H,0,0,0,1,1,4,2,NaN,NaN,NaN,morom001,"['marts002', 'luplj001', 'rodrs002', 'diaze005']",1,0,L,['8'],NaN,24,20,3,0
188902,WAS201709300,77,solis001,5,9,0,bostc001,22,..BBCSX,7/F,0,0,0,1,1,4,3,NaN,NaN,NaN,morom001,"['marts002', 'luplj001', 'rodrs002', 'diaze005']",1,0,F,['7'],NaN,20,25,0,1
188903,WAS201709300,79,rivef001,2,9,1,difow001,1,.SX,8/F,0,0,0,0,0,0,1,NaN,NaN,NaN,NaN,[],1,4,F,['8'],NaN,1,9,0,1
188904,WAS201709300,80,rivef001,7,9,1,murpd006,12,BSSFS,K,0,0,0,0,0,0,2,NaN,NaN,NaN,NaN,[],1,4,NaN,[],NaN,9,17,0,1


In [ ]:
class PybballETL:
    """"""
    def __init__(self):
        None

    def extract_pbb(self):
        team_lkup = pbb.teamid_lookup.team_ids()[['yearID','teamID','teamIDBR','teamIDretro']]
        team_lkup = team_lkup.loc[
            (team_lkup.yearID>=self.season_range[0])
            & (team_lkup.yearID<=self.season_range[1])
        ]
        player_lkup = None
        
        data = []
        for year_team in team_lkup.values:
            try:
                data.append(
                    pbb.schedule_and_record(year_team[0],year_team[2])
                    .assign(SEASON=year_team[0])
                )
            except:
                # print(year_team)
                # raise
                pass

        self.pbb_data = {
            'SCHEDULE_RECORD': pd.concat(data).merge(
                team_lkup,how='inner',
                left_on=['Tm','SEASON'],
                right_on=['teamIDBR','yearID'],
            ),
        }

In [3]:
pbb.teamid_lookup.team_ids()[['yearID','teamID','teamIDBR','teamIDretro']].yearID.max()

2020

In [78]:
class ETL:
    def __init__(self):
        self.SEASON_RANGE = (2017,2017)
        self.GAME_LOOKBACK = [None,3,6,9,12,20,50]
        self.ATBAT_LOOKBACK = [None,20,50,100,300,600]
    

In [5]:
batting

,IDfg,Season,Name,Team,Age,G,AB,PA,H,1B,2B,3B,HR,R,RBI,BB,IBB,SO,HBP,SF,SH,GDP,SB,CS,AVG,GB,FB,LD,IFFB,Pitches,Balls,Strikes,IFH,BU,BUH,BB%,K%,BB/K,OBP,SLG,OPS,ISO,BABIP,GB/FB,LD%,GB%,FB%,IFFB%,HR/FB,IFH%,BUH%,wOBA,wRAA,wRC,Bat,Fld,Rep,Pos,RAR,WAR,Dol,Spd,wRC+,WPA,-WPA,+WPA,RE24,REW,pLI,phLI,PH,WPA/LI,Clutch,FB% (Pitch),FBv,SL%,SLv,CT%,CTv,CB%,CBv,CH%,CHv,SF%,SFv,KN%,KNv,XX%,PO%,wFB,wSL,wCT,wCB,wCH,wSF,wKN,wFB/C,wSL/C,wCT/C,wCB/C,wCH/C,wSF/C,wKN/C,O-Swing%,Z-Swing%,Swing%,O-Contact%,Z-Contact%,Contact%,Zone%,F-Strike%,SwStr%,BsR,FA% (sc),FT% (sc),FC% (sc),FS% (sc),FO% (sc),SI% (sc),SL% (sc),CU% (sc),KC% (sc),EP% (sc),CH% (sc),SC% (sc),KN% (sc),UN% (sc),vFA (sc),vFT (sc),vFC (sc),vFS (sc),vFO (sc),vSI (sc),vSL (sc),vCU (sc),vKC (sc),vEP (sc),vCH (sc),vSC (sc),vKN (sc),FA-X (sc),FT-X (sc),FC-X (sc),FS-X (sc),FO-X (sc),SI-X (sc),SL-X (sc),CU-X (sc),KC-X (sc),EP-X (sc),CH-X (sc),SC-X (sc),KN-X (sc),FA-Z (sc),FT-Z (sc),FC-Z (sc),FS-Z (sc),FO-Z (sc),SI-Z (sc),SL-Z (sc),CU-Z (sc),KC-Z (sc),EP-Z (sc),CH-Z (sc),SC-Z (sc),KN-Z (sc),wFA (sc),wFT (sc),wFC (sc),wFS (sc),wFO (sc),wSI (sc),wSL (sc),wCU (sc),wKC (sc),wEP (sc),wCH (sc),wSC (sc),wKN (sc),wFA/C (sc),wFT/C (sc),wFC/C (sc),wFS/C (sc),wFO/C (sc),wSI/C (sc),wSL/C (sc),wCU/C (sc),wKC/C (sc),wEP/C (sc),wCH/C (sc),wSC/C (sc),wKN/C (sc),O-Swing% (sc),Z-Swing% (sc),Swing% (sc),O-Contact% (sc),Z-Contact% (sc),Contact% (sc),Zone% (sc),Pace,Def,wSB,UBR,Age Rng,Off,Lg,wGDP,Pull%,Cent%,Oppo%,Soft%,Med%,Hard%,TTO%,CH% (pi),CS% (pi),CU% (pi),FA% (pi),FC% (pi),FS% (pi),KN% (pi),SB% (pi),SI% (pi),SL% (pi),XX% (pi),vCH (pi),vCS (pi),vCU (pi),vFA (pi),vFC (pi),vFS (pi),vKN (pi),vSB (pi),vSI (pi),vSL (pi),vXX (pi),CH-X (pi),CS-X (pi),CU-X (pi),FA-X (pi),FC-X (pi),FS-X (pi),KN-X (pi),SB-X (pi),SI-X (pi),SL-X (pi),XX-X (pi),CH-Z (pi),CS-Z (pi),CU-Z (pi),FA-Z (pi),FC-Z (pi),FS-Z (pi),KN-Z (pi),SB-Z (pi),SI-Z (pi),SL-Z (pi),XX-Z (pi),wCH (pi),wCS (pi),wCU (pi),wFA (pi),wFC (pi),wFS (pi),wKN (pi),wSB (pi),wSI (pi),wSL (pi),wXX (pi),wCH/C (pi),wCS/C (pi),wCU/C (pi),wFA/C (pi),wFC/C (pi),wFS/C (pi),wKN/C (pi),wSB/C (pi),wSI/C (pi),wSL/C (pi),wXX/C (pi),O-Swing% (pi),Z-Swing% (pi),Swing% (pi),O-Contact% (pi),Z-Contact% (pi),Contact% (pi),Zone% (pi),Pace (pi),FRM,AVG+,BB%+,K%+,OBP+,SLG+,ISO+,BABIP+,LD+%,GB%+,FB%+,HR/FB%+,Pull%+,Cent%+,Oppo%+,Soft%+,Med%+,Hard%+,EV,LA,Barrels,Barrel%,maxEV,HardHit,HardHit%,Events,CStr%,CSW%,xBA,xSLG,xwOBA,L-WAR
1,18401,2023,Ronald Acuna Jr.,ATL,25,84,337,385,113,67,24,1,21,77,54,42,2,48,4,2,0,7,40,7,0.335,145,90,56,6,1483,598,885,16,0,0,0.109,0.125,0.88,0.413,0.599,1.012,0.264,0.341,0.0161,0.192,0.498,0.309,0.067,0.233,0.110,0.0,0.428,35.0,82,33.2,-0.1,11.9,-3.8,47.6,4.8,$38.1,6.9,170,3.60,-5.15,8.75,41.51,4.28,0.91,NaN,0,3.55,0.42,0.501,93.7,0.233,85.0,0.064,89.3,0.078,80.0,0.096,85.4,0.028,85.0,NaN,NaN,0.008,NaN,17.5,11.2,1.0,5.8,-1.0,2.4,NaN,2.37,3.27,1.08,5.02,-0.67,5.74,NaN,0.271,0.719,0.456,0.736,0.868,0.822,0.414,0.587,0.081,5.3,0.301,NaN,0.062,0.024,0.002,0.194,0.234,0.059,0.020,0.003,0.098,0.001,NaN,NaN,94.0,NaN,89.4,84.7,81.4,94.1,85.0,79.1,81.8,54.2,85.6,81.3,NaN,-2.3,NaN,0.5,-4.9,-6.1,-7.0,1.9,1.8,3.1,-3.9,-1.4,-7.0,NaN,9.1,NaN,4.9,1.8,-0.2,5.2,1.3,-5.5,-5.5,6.9,3.9,-2.6,NaN,13.0,NaN,0.5,1.9,0.0,4.0,9.3,5.3,0.1,0.4,-0.5,0.1,NaN,2.93,NaN,0.52,5.46,0.30,1.40,2.69,6.13,0.39,10.68,-0.35,4.27,NaN,0.235,0.676,0.457,0.669,0.873,0.821,0.504,18.1,-3.9,4.3,1.7,25 - 25,38.4,1.1,-0.7,0.457,0.333,0.210,0.100,0.460,0.440,0.288,0.099,NaN,0.078,0.293,0.060,0.027,NaN,0.001,0.201,0.240,NaN,85.4,NaN,79.6,94.1,89.7,84.8,NaN,81.9,94.4,85.3,NaN,-1.3,NaN,2.9,-1.9,0.8,-4.7,NaN,-6.6,-6.8,2.3,NaN,2.5,NaN,-7.2,8.2,4.0,0.5,NaN,-4.1,4.1,0.1,NaN,-0.4,NaN,3.6,13.0,1.2,2.0,NaN,0.1,3.9,9.9,NaN,-0.25,NaN,3.19,3.03,1.44,4.99,NaN,4.27,1.33,2.83,NaN,0.230,0.673,0.456,0.655,0.878,0.823,0.510,18.1,NaN,134,123,55,128,145,162,114,0.95,116,84,187,112,97,85,64,89,135,94.7,7.8,45,0.155,116.7,159,0.546,291,0.140,0.221,0.353,0.665,0.463,4.7
0,19755,2023,Shohei Ohtani,LAA,28,85,327,382,99,48,15,5,31,61,68,47,4

In [4]:
batting = pbb.batting_stats(2023)

In [ ]:
batting.sort_values(['Name','Season']).groupby(['Season']).count()

In [ ]:
etl.player

In [ ]:
# etl.pbb_data['SCHEDULE_RECORD'].head(20)

In [670]:
pd.DataFrame([[1,2],[3,4]]).to_dict(orient='index')

{0: {0: 1, 1: 2}, 1: {0: 3, 1: 4}}

In [ ]:
etl.extract_pbb()

In [ ]:
# pbb.teamid_lookup.team_ids().loc[pbb.teamid_lookup.team_ids().teamID=='MIA']

In [ ]:
# pbb.schedule_and_record(2021,'MIA')

In [ ]:
etl.team.loc[etl.team.TEAM_ID=='SEA'].sort_values(['TEAM_ID','SEASON','DATE']).head(20)

In [ ]:
etl.rs_batting.stat.unique()

In [ ]:
etl.game

In [ ]:
etl.game

In [ ]:
etl.rs_lineup

In [ ]:
data = pbb.statcast(start_dt='2017-06-24', end_dt='2017-06-27')
# data = pbb.schedule_and_record(1927, 'NYY')
data.head()

In [ ]:
pbb.playerid_lookup('judge', 'aaron')

In [ ]:
df = pbb.schedule_and_record(2021,'WSN')

In [ ]:
mo_map = {
    'JAN': 1,'FEB': 2,'MAR': 3,'APR': 4,
    'MAY': 5,'JUN': 6,'JUL': 7,'AUG': 8,
    'SEP': 9,'OCT': 10,'NOV': 11,'DEC': 12,
}
df['DATE'] = df.Date.map(
    lambda x: datetime.date(
        2021,
        mo_map[x.split(',')[1].split(' ')[1].strip().upper()],
        int(x.split(',')[1].split(' ')[2].strip()),
    )
)

In [ ]:
df.head(20)